# Nhóm 2, Notebook_B (Sinh viên B): mô hình, khoảng dự báo, dịch chuyển phân phối

**Đề tài**: Khoảng dự báo có cam kết bao phủ cho dung lượng pin xe điện, khi đổi xe và khi đổi thời kỳ


**Vai trò**: Sinh viên B nhận `sessions.csv` từ Sinh viên A, so sánh mô hình, dựng khoảng dự báo
conformal, đo mức bao phủ dưới hai trục dịch chuyển, và trả lời câu hỏi chính của bài:
**cần bao nhiêu phiên sạc có nhãn của một chiếc xe mới thì khoảng dự báo lấy lại cam kết bao phủ?**

**Đầu vào**: `ev_data/data/sessions.csv`, `feature_names.csv`, `manifest.json`
**Đầu ra**: các bảng và hình trong `results/` và `fig/` để đưa thẳng vào bài báo.
**Thời gian chạy đo được**: khoảng 110 phút trên 2 lõi CPU với bộ 167 đặc trưng, hoặc khoảng 55
phút nếu đặt `BO = 'bai_bao'` để chạy trên bộ 43. Đặt `NHANH = True` để thử với 3 xe trước.

## Câu hỏi:

Mô hình đoán pack còn 118 Ah. Câu đó vô dụng nếu không kèm sai số. Ta muốn nói:
*"còn 118 Ah, và với xác suất 90 phần trăm sự thật nằm trong khoảng 115 tới 121"*.

Conformal prediction cho ta đúng câu đó, **có chứng minh, trên mẫu hữu hạn, không giả định gì về
mô hình**. Đổi lại nó đòi một điều: tập hiệu chuẩn và điểm cần dự báo phải **hoán vị được**
(exchangeable). Một đội xe phá vỡ điều đó theo hai cách cùng lúc:

| Trục dịch chuyển | Vì sao hoán vị được bị phá | Đo bằng cách nào |
|---|---|---|
| **Đổi xe** | chiếc xe đang xét không có trong tập hiệu chuẩn | giữ lại từng xe, 20 lượt |
| **Đổi thời kỳ** | xe được xét nhiều tháng sau khi thu dữ liệu hiệu chuẩn | huấn luyện trên 75 phần trăm đầu theo lịch |

Notebook này đo cả hai, rồi đo cách sửa.

## Thứ tự các bước

Bước 0 nạp dữ liệu → Bước 1 tám mô hình → Bước 2 ba giao thức chia dữ liệu →
Bước 3 công thức conformal → Bước 4 phép thử bắt buộc khi hoán vị được →
**Bước 5 vòng lặp chính, đo sai số điểm và bao phủ cùng một lúc** →
Bước 6 kiểm định thống kê → Bước 7 ngân sách hiệu chuẩn → Bước 8 trọng số dịch chuyển →
Bước 9 xuất bảng và hình.

Bước 5 là chỗ tốn thời gian nhất. Nó khớp mỗi mô hình **đúng một lần** cho mỗi lượt giữ xe rồi
lấy ra cả hai thứ ta cần. Nếu tách thành hai vòng lặp thì thời gian chạy tăng gấp đôi mà không
thêm thông tin gì, đây là lỗi mà bản nháp đầu của notebook này đã mắc.

## Vừa làm vừa hỏi AI và ghi Audit Log

Giống Notebook A. Riêng Notebook B có một cái bẫy mà AI hay trả lời sai, nên ghi rõ ở đây:

> Nếu bạn hỏi AI *"tính khoảng conformal giúp tôi"*, rất nhiều câu trả lời sẽ dùng
> `np.quantile(scores, 0.9)`. **Sai.** Mức phân vị đúng là $\lceil (n+1)(1-\alpha)\rceil / n$,
> và phải lấy `method='higher'`. Với $n$ lớn thì hai cách gần bằng nhau, với $n$ nhỏ thì lệch hẳn,
> và cả bài này xoay quanh trường hợp $n$ nhỏ.

| Ngày, bước | Prompt (rút gọn) | AI trả lời gì | Tôi kiểm tra hoặc sửa gì | Dùng vào đâu |
|---|---|---|---|---|
| 20/09, Bước 3 | tính khoảng conformal 90 phần trăm | dùng np.quantile(s, 0.9) | đối chiếu Lei 2018, sửa thành ceil((n+1)*0.9)/n và method='higher' | Bước 3, hàm `q_level` |

## Bước 0: nạp dữ liệu Sinh viên A bàn giao

Kiểm mã băm trước khi làm gì khác. Nếu không khớp với `manifest.json`, bạn đang cầm bản cũ.

In [1]:
import json, pathlib, hashlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 9, 'axes.grid': True, 'grid.alpha': .25})
TEAL, RUST, SAND, SLATE = '#0F3D3E', '#B5533C', '#C9A227', '#4B6A8A'

SEED = 42
ALPHA = 0.10                     # muốn bao phủ 90 phần trăm
NOM = 1 - ALPHA
np.random.seed(SEED)

DATA = pathlib.Path('ev_data/data')
RES = pathlib.Path('results'); RES.mkdir(exist_ok=True)
FIG = pathlib.Path('fig'); FIG.mkdir(exist_ok=True)

d = pd.read_csv(DATA / 'sessions.csv', parse_dates=['t_start'])
man = json.loads((DATA / 'manifest.json').read_text())
h = hashlib.sha256((DATA / 'sessions.csv').read_bytes()).hexdigest()[:16]
print('mã băm khớp manifest:', h == man['sha256_16'])

META = ['vehicle', 'session', 't_start', 'n_rows', 'ah', 'capacity', 'bms_cap']
FEATS_ALL = [c for c in d.columns if c not in META]
FEATS_PAPER = pd.read_csv(DATA / 'feature_names_paper.csv', header=None)[0].tolist()
assert set(FEATS_PAPER) <= set(FEATS_ALL), 'bộ 43 không nằm trong bộ mở rộng'

BO = 'mo_rong'                   # 'mo_rong' cho bài của sinh viên, 'bai_bao' để tái hiện bài báo
FEATS = FEATS_ALL if BO == 'mo_rong' else FEATS_PAPER

d = d.dropna(subset=['capacity']).reset_index(drop=True)
print(f"{len(d):,} phiên, {d.vehicle.nunique()} xe")
print(f"bộ mở rộng {len(FEATS_ALL)} đặc trưng, bộ của bài báo {len(FEATS_PAPER)}")
print(f"đang dùng bộ '{BO}' với {len(FEATS)} đặc trưng")
print(f"từ {d.t_start.min():%m/%Y} tới {d.t_start.max():%m/%Y}")
print(f"dung lượng: trung vị {d.capacity.median():.1f} Ah, "
      f"độ lệch chuẩn {d.capacity.std():.1f} Ah")

mã băm khớp manifest: True
31,253 phiên, 20 xe
bộ mở rộng 167 đặc trưng, bộ của bài báo 43
đang dùng bộ 'mo_rong' với 167 đặc trưng
từ 07/2019 tới 11/2021
dung lượng: trung vị 122.2 Ah, độ lệch chuẩn 6.1 Ah


## Bước 1: tám mô hình

Sáu mô hình học máy, cộng hai mốc so sánh không cần học:

| Mô hình | Vai trò |
|---|---|
| Trung bình | sàn tuyệt đối, đoán cùng một số cho mọi phiên |
| BMS thô | con số chính chiếc xe đã tự tính, miễn phí |
| BMS hiệu chỉnh | BMS sau khi khử độ lệch hệ thống bằng hồi quy tuyến tính trên tập huấn luyện |
| Ridge, Elastic net | tuyến tính, nhanh, dễ giải thích |
| Rừng ngẫu nhiên, XGBoost | phi tuyến, thường mạnh nhất trên bảng số |
| Mạng nơ ron | để có mặt một họ khác |

**Vì sao phải có BMS hiệu chỉnh**: BMS thô lệch hệ thống khoảng 6 Ah so với nhãn tích phân. So với
BMS thô là so với một đối thủ bị trói tay. Hiệu chỉnh xong mới là so công bằng, và mới đo được
*đường cong sạc đáng giá bao nhiêu*.

In [3]:
from sklearn.linear_model import RidgeCV, ElasticNetCV
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              HistGradientBoostingClassifier)
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor

CAP_LO, CAP_HI = 60.0, 220.0

def tao_mo_hinh(seed):
    return {
        'Trung bình': 'mean',
        'BMS thô': 'bms',
        'BMS hiệu chỉnh': 'bms_fit',
        'Ridge': make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 4, 40))),
        'Elastic net': make_pipeline(StandardScaler(),
                                     ElasticNetCV(l1_ratio=[.1, .5, .9, 1.], cv=4,
                                                  max_iter=20000, n_jobs=2, random_state=seed)),
        'Rừng ngẫu nhiên': RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                                 max_features=.4, n_jobs=2, random_state=seed),
        'XGBoost': XGBRegressor(n_estimators=600, max_depth=6, learning_rate=.05,
                                subsample=.8, colsample_bytree=.7, min_child_weight=5,
                                reg_lambda=2., tree_method='hist', n_jobs=2, random_state=seed),
        'Mạng nơ ron': make_pipeline(StandardScaler(),
                                     MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=600,
                                                  alpha=1e-3, learning_rate_init=2e-3,
                                                  early_stopping=True, n_iter_no_change=20,
                                                  random_state=seed)),
    }

TEN_MO_HINH = list(tao_mo_hinh(0).keys())

def cot_bms(tr, q):
    p = q.bms_cap.values.astype(float).copy()
    p[~np.isfinite(p)] = tr.capacity.mean()
    return p

def du_bao(ten, seed, tr, cac_tap, feats=None):
    feats = feats or FEATS
    m = tao_mo_hinh(seed)[ten]
    if m == 'mean':
        mu = tr.capacity.mean()
        return [np.full(len(q), mu) for q in cac_tap]
    if m == 'bms':
        return [np.clip(cot_bms(tr, q), CAP_LO, CAP_HI) for q in cac_tap]
    if m == 'bms_fit':
        b, a = np.polyfit(cot_bms(tr, tr), tr.capacity.values, 1)
        return [np.clip(a + b * cot_bms(tr, q), CAP_LO, CAP_HI) for q in cac_tap]
    m.fit(tr[feats].values, tr.capacity.values)
    return [np.clip(m.predict(q[feats].values), CAP_LO, CAP_HI) for q in cac_tap]

def diem(y, p):
    e = np.asarray(p) - np.asarray(y)
    return dict(RMSE=float(np.sqrt(np.mean(e**2))), MAE=float(np.mean(np.abs(e))),
                MAPE=float(100*np.mean(np.abs(e)/y)),
                R2=float(1 - np.sum(e**2)/np.sum((y - np.mean(y))**2)))

print(f'{len(TEN_MO_HINH)} mô hình đã định nghĩa')

8 mô hình đã định nghĩa


## Bước 1b: bộ mở rộng có đáng không, và đặc trưng nào mang thông tin

Sinh viên A bàn giao 167 đặc trưng, trong đó 43 là bộ của bài báo. Trước khi chạy thí nghiệm dài,
trả lời hai câu:

1. **Thêm 124 đặc trưng có làm mô hình tốt lên không?** Nếu không, bài của bạn có ngay một kết quả
   đáng viết: thông tin đã bão hoà ở bộ nhỏ.
2. **Nhóm nào mang thông tin?** Đây là chỗ để nói chuyện vật lý chứ không chỉ chuyện số.

Ô này chạy khoảng 2 phút.

In [4]:
from sklearn.model_selection import train_test_split

tr0, te0 = train_test_split(d, test_size=0.3, random_state=SEED)
so_sanh = []
for ten_bo, cols in [('bài báo, 43', FEATS_PAPER), ('mở rộng, 167', FEATS_ALL)]:
    (p0,) = du_bao('XGBoost', 0, tr0, [te0], cols)
    so_sanh.append({'bộ': ten_bo, 'số đặc trưng': len(cols), **diem(te0.capacity.values, p0)})
bs = pd.DataFrame(so_sanh)
bs.to_csv(RES / 'so_sanh_bo_dac_trung.csv', index=False)
print(bs.round(4).to_string(index=False))

m = tao_mo_hinh(0)['XGBoost']
m.fit(tr0[FEATS_ALL].values, tr0.capacity.values)
imp = pd.Series(m.feature_importances_, index=FEATS_ALL).sort_values(ascending=False)
imp.to_csv(RES / 'tam_quan_trong.csv')

nhom = imp.groupby([c.split('_')[0] for c in imp.index]).sum().sort_values(ascending=False)
print('\n20 đặc trưng quan trọng nhất:')
print(imp.head(20).round(4).to_string())
print('\ntổng tầm quan trọng theo nhóm tiền tố:')
print(nhom.round(4).to_string())
print(f"\ntrong 20 đặc trưng đầu, có {sum(c in FEATS_PAPER for c in imp.head(20).index)} "
      f"thuộc bộ của bài báo")

          bộ  số đặc trưng   RMSE    MAE   MAPE     R2
 bài báo, 43            43 1.9886 1.3827 1.1423 0.8943
mở rộng, 167           167 1.9021 1.3040 1.0784 0.9033

20 đặc trưng quan trọng nhất:
i_min          0.1165
i_a10          0.0974
i_end          0.0363
v_end          0.0276
vcell_range    0.0234
ic_b7          0.0190
vcell_max      0.0188
dv_area        0.0186
i_q10          0.0178
spread_p10     0.0168
dv_q4          0.0165
ic_b5          0.0148
soc_hi         0.0136
dv_q3          0.0126
ic_b9          0.0125
ic_b6          0.0125
spread_max     0.0117
dv_d9          0.0116
i_q50          0.0113
vmin_p10       0.0107

tổng tầm quan trọng theo nhóm tiền tố:
i         0.3216
vcell     0.1331
dv        0.1123
ic        0.0839
spread    0.0785
v         0.0596
vmin      0.0431
tmax      0.0420
vmax      0.0382
tmin      0.0278
soc       0.0272
vpack     0.0143
d2v       0.0057
r         0.0048
iamp      0.0041
tgrad     0.0029
c         0.0008

trong 20 đặc trưng đầu, có 7 thuộc

### Đọc kết quả trên

Nếu RMSE của bộ 167 **không** thấp hơn bộ 43 một cách rõ rệt, đừng buồn và cũng đừng giấu. Đó là một
phát hiện: đường cong sạc chỉ mang chừng đó thông tin, và thêm cách mô tả nó không thêm thông tin.
Hãy viết đúng như vậy trong bài, kèm bảng này.

Nếu bộ 167 tốt hơn, hãy tìm xem **nhóm nào** đóng góp, rồi giải thích bằng vật lý pin chứ không chỉ
bằng con số. Ví dụ nhóm `ic` là đường cong dung lượng vi phân, vốn là dấu hiệu lão hoã kinh điển
trong tài liệu; nếu nó lên cao thì đó là một đoạn hay để viết.

## Bước 2: ba giao thức chia dữ liệu

Đây là phần dễ làm sai nhất và cũng quyết định toàn bộ kết luận của bài.

| Giao thức | Chia thế nào | Hoán vị được không | Đo cái gì |
|---|---|---|---|
| **Xáo trộn** | trộn ngẫu nhiên, 60/20/20 | **có** | mốc lạc quan, và là phép thử xem code conformal có đúng không |
| **Đổi xe** | giữ lại trọn một xe, lặp 20 lượt | không | xe mới chưa từng thấy |
| **Đổi thời kỳ** | sắp theo ngày, 60 phần trăm đầu huấn luyện, 20 phần trăm cuối kiểm tra | không | triển khai hôm nay, dùng cho năm sau |

**Bẫy**: nếu bạn chia ngẫu nhiên trong khi dữ liệu có cấu trúc theo xe hoặc theo thời gian, bạn đang
đo giao thức xáo trộn và tưởng mình đo giao thức kia. Kết quả sẽ đẹp và sai.

In [5]:
def chia(df, rng=None, fr=(.6, .2)):
    n = len(df)
    idx = np.arange(n) if rng is None else rng.permutation(n)
    a, b = int(fr[0]*n), int((fr[0]+fr[1])*n)
    return df.iloc[idx[:a]], df.iloc[idx[a:b]], df.iloc[idx[b:]]

ds = d.sort_values('t_start')
tr_t, cal_t, te_t = chia(ds)
print('giao thức đổi thời kỳ:')
for ten, x in [('huấn luyện', tr_t), ('hiệu chuẩn', cal_t), ('kiểm tra', te_t)]:
    print(f'  {ten:11s} {x.t_start.min():%m/%Y} tới {x.t_start.max():%m/%Y}  '
          f'n={len(x):6,}  dung lượng trung vị {x.capacity.median():.1f} Ah')
print('\n-> pack đã suy giảm khoảng '
      f'{tr_t.capacity.median()-te_t.capacity.median():.0f} Ah giữa hai thời kỳ. '
      'Đây là dịch chuyển thật, không phải nhiễu.')

giao thức đổi thời kỳ:
  huấn luyện  07/2019 tới 12/2020  n=18,751  dung lượng trung vị 125.8 Ah
  hiệu chuẩn  12/2020 tới 05/2021  n= 6,251  dung lượng trung vị 119.1 Ah
  kiểm tra    05/2021 tới 11/2021  n= 6,251  dung lượng trung vị 116.1 Ah

-> pack đã suy giảm khoảng 10 Ah giữa hai thời kỳ. Đây là dịch chuyển thật, không phải nhiễu.


## Bước 3: conformal, phần cốt lõi

Cho mô hình $\hat\mu$ khớp trên tập huấn luyện, và tập hiệu chuẩn $\{(x_i,y_i)\}_{i=1}^n$ **tách rời**
tập huấn luyện. Với một điểm số phù hợp $s(x,y)$, đặt

$$\hat q \;=\; \text{phân vị mức } \frac{\lceil (n+1)(1-\alpha)\rceil}{n} \text{ của } \{s(x_i,y_i)\}$$

thì khoảng $\{y : s(x,y) \le \hat q\}$ chứa sự thật với xác suất ít nhất $1-\alpha$.

**Ba chỗ dễ sai, kiểm cả ba trước khi tin kết quả:**

| Sai lầm | Hậu quả | Cách kiểm |
|---|---|---|
| tập hiệu chuẩn trùng tập huấn luyện | bao phủ sụp còn vài phần trăm | `assert` hai tập rời nhau |
| dùng `np.quantile(s, 0.9)` | thiếu bao phủ khi $n$ nhỏ | so với `q_level(n)` ở $n=10$ |
| mức phân vị vượt 1,0 khi $n$ quá nhỏ | khoảng rộng vô lý | kẹp về 1,0 và ghi chú |

Ba điểm số phù hợp được so sánh:

- **Sai số tuyệt đối**: $s=|y-\hat\mu(x)|$, khoảng rộng như nhau ở mọi điểm.
- **Sai số chuẩn hoá**: $s=|y-\hat\mu(x)|/\hat\sigma(x)$, khoảng rộng ra ở chỗ mô hình tự thấy khó.
- **CQR**: conformal hoá hai hồi quy phân vị, cho khoảng bất đối xứng.

In [6]:
def q_level(n, a=ALPHA):
    # Mức phân vị hữu hạn mẫu. KHÔNG phải 1-alpha.
    return min(1.0, np.ceil((n + 1) * (1 - a)) / n)

print('n=10  ->', round(q_level(10), 4), ' (np.quantile ngây thơ sẽ dùng 0.9)')
print('n=50  ->', round(q_level(50), 4))
print('n=2000->', round(q_level(2000), 4))
print('n=5   ->', round(q_level(5), 4), ' <- đã kẹp về 1.0, khoảng sẽ rất rộng, đó là trung thực')

def khoang_tuyet_doi(res_cal, p_te, a=ALPHA):
    q = np.quantile(np.abs(res_cal), q_level(len(res_cal), a), method='higher')
    return p_te - q, p_te + q

def khoang_chuan_hoa(res_cal, sig_cal, p_te, sig_te, a=ALPHA):
    s = np.abs(res_cal) / np.maximum(sig_cal, 1e-6)
    q = np.quantile(s, q_level(len(s), a), method='higher')
    h = q * np.maximum(sig_te, 1e-6)
    return p_te - h, p_te + h

def khoang_cqr(lo_cal, hi_cal, y_cal, lo_te, hi_te, a=ALPHA):
    s = np.maximum(lo_cal - y_cal, y_cal - hi_cal)
    q = np.quantile(s, q_level(len(s), a), method='higher')
    return lo_te - q, hi_te + q

def bao_phu(lo, hi, y):
    lo, hi, y = map(np.asarray, (lo, hi, y))
    return (float(((y >= lo) & (y <= hi)).mean()),
            float(np.median(hi - lo)),
            float(np.median((hi - lo) / y) * 100))

n=10  -> 1.0  (np.quantile ngây thơ sẽ dùng 0.9)
n=50  -> 0.92
n=2000-> 0.9005
n=5   -> 1.0  <- đã kẹp về 1.0, khoảng sẽ rất rộng, đó là trung thực


## Bước 4: phép thử bắt buộc, giao thức xáo trộn

Ở giao thức xáo trộn, hoán vị được **đúng theo thiết kế**, nên bao phủ đo được phải rơi rất sát 0,90.
Nếu không, code của bạn sai, chứ không phải lý thuyết sai. Đây là phép thử phải qua trước khi làm tiếp.

Ô này khớp thêm hai mô hình phụ: một rừng ngẫu nhiên đoán *độ khó* của từng phiên cho điểm số chuẩn
hoá, và hai hồi quy phân vị cho CQR. Mất khoảng 3 phút.

In [7]:
def khoi_conformal(tr, cal, te, seed=0):
    p_cal, p_te, p_tr = du_bao('XGBoost', seed, tr, [cal, te, tr])
    sig = RandomForestRegressor(n_estimators=150, min_samples_leaf=10, max_features=.4,
                                n_jobs=2, random_state=seed)
    sig.fit(tr[FEATS].values, np.log(np.abs(tr.capacity.values - p_tr) + 1e-3))
    qlo = GradientBoostingRegressor(loss='quantile', alpha=ALPHA/2, n_estimators=200,
                                    max_depth=4, learning_rate=.08, subsample=.8,
                                    random_state=seed).fit(tr[FEATS].values, tr.capacity.values)
    qhi = GradientBoostingRegressor(loss='quantile', alpha=1-ALPHA/2, n_estimators=200,
                                    max_depth=4, learning_rate=.08, subsample=.8,
                                    random_state=seed).fit(tr[FEATS].values, tr.capacity.values)
    return dict(p_cal=p_cal, p_te=p_te,
                s_cal=np.exp(sig.predict(cal[FEATS].values)),
                s_te=np.exp(sig.predict(te[FEATS].values)),
                lo_cal=qlo.predict(cal[FEATS].values), hi_cal=qhi.predict(cal[FEATS].values),
                lo_te=qlo.predict(te[FEATS].values), hi_te=qhi.predict(te[FEATS].values))

rows = []
for s in range(2):
    tr, cal, te = chia(d, np.random.default_rng(s))
    assert len(set(tr.index) & set(cal.index)) == 0, 'huấn luyện và hiệu chuẩn bị trùng'
    b = khoi_conformal(tr, cal, te, s)
    r = cal.capacity.values - b['p_cal']
    for ten, (lo, hi) in [
            ('Sai số tuyệt đối', khoang_tuyet_doi(r, b['p_te'])),
            ('Sai số chuẩn hoá', khoang_chuan_hoa(r, b['s_cal'], b['p_te'], b['s_te'])),
            ('CQR', khoang_cqr(b['lo_cal'], b['hi_cal'], cal.capacity.values,
                               b['lo_te'], b['hi_te']))]:
        c, w, rw = bao_phu(lo, hi, te.capacity.values)
        rows.append({'score': ten, 'seed': s, 'bao_phu': c, 'do_rong': w, 'rong_pct': rw})
    print(f'seed {s} xong', flush=True)

ind = pd.DataFrame(rows)
ind.to_csv(RES / 'conformal_xaotron.csv', index=False)
print()
print(ind.groupby('score')[['bao_phu', 'do_rong', 'rong_pct']].mean().round(4).to_string())
assert abs(ind.bao_phu.mean() - NOM) < 0.02, 'bao phủ lệch quá xa 0,90, rà lại code conformal'
print('\nphép thử đạt: bao phủ bám sát mức danh nghĩa khi hoán vị được')

seed 0 xong
seed 1 xong

                  bao_phu  do_rong  rong_pct
score                                       
CQR                0.9064   8.2619    6.7506
Sai số chuẩn hoá   0.8976   5.6086    4.6123
Sai số tuyệt đối   0.9000   5.9111    4.8384

phép thử đạt: bao phủ bám sát mức danh nghĩa khi hoán vị được


## Bước 5: vòng lặp chính

Đây là ô tốn thời gian nhất, khoảng 95 phút với bộ 167 đặc trưng. Nó làm **ba giao thức trong một
vòng lặp** và với mỗi
lần khớp mô hình lấy ra **cả hai** thứ ta cần:

1. sai số điểm trên tập kiểm tra;
2. bao phủ của khoảng conformal, hiệu chuẩn bằng tập hiệu chuẩn của miền nguồn.

Với giao thức đổi xe, ta còn giữ lại dự báo của XGBoost cho từng xe, vì Bước 7 và Bước 8 cần đúng
những dự báo đó. Khớp lại từ đầu ở mỗi bước là tự nhân đôi thời gian chạy mà không thêm thông tin.

Đặt `NHANH = True` để chạy thử với 3 xe trước khi chạy đủ.

In [8]:
NHANH = True                  # True: chỉ 3 lượt giữ xe, để thử code
SEEDS = [0, 1, 2]
N_SRC = 2000                       # số phiên nguồn dùng để hiệu chuẩn

diem_rows, cov_rows = [], []
giu_lai = {}                       # dự báo XGBoost của từng xe, dùng lại ở Bước 7 và 8

def mot_luot(tr, cal, te, giao_thuc, fold, seed=0, giu=False):
    for m in TEN_MO_HINH:
        pc, pt = du_bao(m, seed, tr, [cal, te])
        diem_rows.append({'giao_thuc': giao_thuc, 'model': m, 'fold': fold,
                          **diem(te.capacity.values, pt)})
        lo, hi = khoang_tuyet_doi(cal.capacity.values - pc, pt)
        c, w, _ = bao_phu(lo, hi, te.capacity.values)
        cov_rows.append({'giao_thuc': giao_thuc, 'model': m, 'fold': fold,
                         'bao_phu': c, 'do_rong': w})
        if giu and m == 'XGBoost':
            giu_lai[fold] = (cal, pc, te, pt)

# --- giao thức xáo trộn
for s in SEEDS:
    tr, cal, te = chia(d, np.random.default_rng(s))
    mot_luot(tr, cal, te, 'xáo trộn', s, s)
    print(f'xáo trộn seed {s} xong', flush=True)

# --- giao thức đổi xe, 20 lượt giữ xe
xe_list = sorted(d.vehicle.unique())[:3] if NHANH else sorted(d.vehicle.unique())
for v in xe_list:
    src, tgt = d[d.vehicle != v], d[d.vehicle == v].reset_index(drop=True)
    i = np.random.default_rng(v).permutation(len(src))
    tr = src.iloc[i[:int(.75*len(src))]]
    cal = src.iloc[i[int(.75*len(src)):]].sample(N_SRC, random_state=0)
    mot_luot(tr, cal, tgt, 'đổi xe', int(v), 0, giu=True)
    print(f'giữ lại xe {v}', flush=True)

# --- giao thức đổi thời kỳ
mot_luot(tr_t, cal_t, te_t, 'đổi thời kỳ', 0, 0)

kq = pd.DataFrame(diem_rows); kq.to_csv(RES / 'diem_so.csv', index=False)
cx = pd.DataFrame(cov_rows);  cx.to_csv(RES / 'conformal_doixe.csv', index=False)

print('\nRMSE (Ah) theo giao thức:')
print(kq.pivot_table(index='model', columns='giao_thuc', values='RMSE')
        .loc[TEN_MO_HINH].round(3).to_string())

KeyboardInterrupt: 

### Đọc bảng trên

Hai điều cần nhận ra, và cả hai đều đi vào phần Kết quả của bài báo:

1. **Đổi xe gần như không làm mô hình tệ đi.** Hai mươi chiếc cùng đời, cùng chạy một thành phố,
   nên đường cong sạc của chúng giống nhau. Nếu bạn chỉ nhìn RMSE, bạn sẽ kết luận "không có dịch
   chuyển đáng kể" và dừng lại ở đó. Bảng bao phủ ngay dưới đây cho thấy kết luận đó thiếu.
2. **Đổi thời kỳ thì làm mô hình tệ hẳn.** Pack đã suy giảm xuống dưới mọi giá trị từng có trong tập
   huấn luyện, nên mô hình phải ngoại suy. $R^2$ âm là bình thường ở đây và cần giải thích trong bài
   chứ không giấu đi.

In [2]:
tt = (cx[cx.giao_thuc == 'đổi xe']
      .groupby('model').agg(bao_phu_tb=('bao_phu', 'mean'),
                            xe_te_nhat=('bao_phu', 'min'),
                            so_xe_duoi=('bao_phu', lambda s: int((s < NOM).sum())),
                            do_rong=('do_rong', 'mean')).loc[TEN_MO_HINH])
print('bao phủ khi giữ lại một xe, hiệu chuẩn bằng 19 xe kia:')
print(tt.round(4).to_string())
print('\nbao phủ khi đổi thời kỳ:')
print(cx[cx.giao_thuc == 'đổi thời kỳ'].set_index('model')
      .loc[TEN_MO_HINH, ['bao_phu', 'do_rong']].round(4).to_string())
print('\n-> trung bình toàn đội xe trông ổn, nhưng cột xe_te_nhat mới là thứ '
      'người dùng cuối nhận được.')

NameError: name 'cx' is not defined

## Bước 6: kiểm định thống kê

20 lượt giữ xe là **mẫu ghép cặp**: cùng một tập xe, chỉ khác mô hình. Nên dùng Friedman để xếp
hạng, rồi Wilcoxon ghép cặp có hiệu chỉnh Holm để so từng mô hình với mô hình dẫn đầu.

In [ ]:
from scipy import stats

pv = kq[kq.giao_thuc == 'đổi xe'].pivot(index='fold', columns='model', values='RMSE')
chi, p = stats.friedmanchisquare(*[pv[m].values for m in pv.columns])
ranks = pv.rank(axis=1).mean().sort_values()
best = ranks.index[0]
print(f'Friedman chi2={chi:.2f}, p={p:.2e} -> thứ hạng khác nhau có ý nghĩa')
print(f'mô hình dẫn đầu theo hạng trung bình: {best}\n')

out = []
for m in pv.columns:
    if m == best:
        out.append({'model': m, 'hang_tb': ranks[m], 'RMSE_tb': pv[m].mean(), 'p_raw': np.nan})
        continue
    _, pp = stats.wilcoxon(pv[best].values, pv[m].values)
    out.append({'model': m, 'hang_tb': ranks[m], 'RMSE_tb': pv[m].mean(), 'p_raw': pp})

sub = sorted([o for o in out if not np.isnan(o['p_raw'])], key=lambda o: o['p_raw'])
prev = 0.0
for i, o in enumerate(sub):                       # hiệu chỉnh Holm
    prev = o['p_holm'] = min(1.0, max(prev, (len(sub)-i) * o['p_raw']))
    o['ket_luan'] = 'kém hơn rõ rệt' if o['p_holm'] < .05 else 'không tách được'
bang_tk = pd.DataFrame(out).sort_values('hang_tb')
bang_tk.to_csv(RES / 'kiem_dinh.csv', index=False)
print(bang_tk.round(4).to_string(index=False))

## Bước 7: ngân sách hiệu chuẩn, thí nghiệm chính của bài

Với mỗi trục dịch chuyển, ta thử ba cách chọn tập hiệu chuẩn:

| Cách | Tập hiệu chuẩn gồm | Tình huống thực tế |
|---|---|---|
| **Chỉ nguồn** | 2000 phiên từ miền nguồn | ngày đầu tiên của một chiếc xe mới |
| **Trộn** | 2000 phiên nguồn + $k$ phiên có nhãn của miền đích | sau vài tuần vận hành |
| **Chỉ đích** | đúng $k$ phiên của miền đích | bỏ hết dữ liệu cũ |

$k$ chạy qua 10, 25, 50, 100, 200. Mỗi $k$ bốc ngẫu nhiên 20 lần để con số không phải may rủi.

Ô này **không khớp lại mô hình nào**, nó dùng lại dự báo đã giữ ở Bước 5, nên chạy trong vài giây.

**Dự đoán trước khi chạy** (ghi vào Audit Log rồi mới chạy, đây là cách tự kiểm tính trung thực):
cách "chỉ đích" với $k=10$ sẽ cho bao phủ cao nhưng khoảng rộng vô lý, vì mức phân vị bị kẹp về 1,0.

In [ ]:
LAP, NGAN_SACH = 20, [10, 25, 50, 100, 200]

def quet_ngan_sach(cal_src, p_src, tgt, p_tgt, ten_truc, fold):
    out = []
    rng = np.random.default_rng(1000 + fold)
    r_src = cal_src.capacity.values - p_src
    y = tgt.capacity.values
    for rep in range(LAP):
        perm = rng.permutation(len(tgt))
        for k in NGAN_SACH:
            if len(tgt) < k + 200:
                continue
            ci, ti = perm[:k], perm[k:]
            r_tgt = y[ci] - p_tgt[ci]
            for ten, r in [('Chỉ nguồn', r_src), ('Trộn', np.r_[r_src, r_tgt]),
                           ('Chỉ đích', r_tgt)]:
                if ten == 'Chỉ nguồn' and rep > 0:
                    continue
                lo, hi = khoang_tuyet_doi(r, p_tgt[ti])
                c, w, _ = bao_phu(lo, hi, y[ti])
                out.append({'truc': ten_truc, 'fold': fold, 'cach': ten, 'k': k,
                            'rep': rep, 'bao_phu': c, 'do_rong': w})
    return out

rows = []
for v, (cal_src, p_src, tgt, p_tgt) in giu_lai.items():
    rows += quet_ngan_sach(cal_src, p_src, tgt, p_tgt, 'đổi xe', v)

# trục thời gian: dùng lại tập huấn luyện và hiệu chuẩn của giao thức đổi thời kỳ
p_ct, p_tt = du_bao('XGBoost', 0, tr_t, [cal_t, te_t.reset_index(drop=True)])
rows += quet_ngan_sach(cal_t, p_ct, te_t.reset_index(drop=True), p_tt, 'đổi thời kỳ', 0)

ns = pd.DataFrame(rows); ns.to_csv(RES / 'ngan_sach.csv', index=False)
print(ns.groupby(['truc', 'cach', 'k'])[['bao_phu', 'do_rong']].mean().round(3).to_string())

### Hình chính của bài báo

In [ ]:
mau = {'Chỉ nguồn': SLATE, 'Trộn': TEAL, 'Chỉ đích': RUST}
fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
for a, truc, ttl in [(ax[0], 'đổi xe', '(a) xe chưa từng thấy'),
                     (ax[1], 'đổi thời kỳ', '(b) thời kỳ về sau')]:
    sub = ns[ns.truc == truc]
    for c, g in sub.groupby('cach'):
        m = g.groupby(['k', 'fold']).bao_phu.mean().groupby('k').agg(['mean', 'std'])
        a.errorbar(m.index, m['mean'], yerr=m['std'].fillna(0), marker='o', ms=4,
                   lw=1.4, capsize=2, color=mau[c], label=c)
    a.axhline(NOM, color='k', ls='--', lw=1)
    a.set_xscale('log'); a.set_xlabel('số phiên có nhãn của miền đích, $k$'); a.set_title(ttl)
ax[0].set_ylabel('bao phủ đo được'); ax[0].legend(loc='lower right')

sub = ns[ns.truc == 'đổi xe']
for c, g in sub.groupby('cach'):
    m = g.groupby(['k', 'fold']).do_rong.mean().groupby('k').agg(['mean', 'std'])
    ax[2].errorbar(m.index, m['mean'], yerr=m['std'].fillna(0), marker='o', ms=4, lw=1.4,
                   capsize=2, color=mau[c], label=c)
ax[2].set_xscale('log'); ax[2].set_xlabel('$k$'); ax[2].set_ylabel('độ rộng trung vị (Ah)')
ax[2].set_title('(c) cái giá của bao phủ')
fig.tight_layout(); fig.savefig(FIG / 'hinh_ngan_sach.png', dpi=300, bbox_inches='tight'); plt.show()

## Bước 8: trọng số dịch chuyển hiệp biến, cách không cần nhãn

Nếu dịch chuyển chỉ nằm ở đầu vào, nghĩa là $p(y|x)$ không đổi, thì có thể sửa bao phủ bằng cách
**gán trọng số** cho từng điểm hiệu chuẩn theo tỉ số $w(x)=p_{\text{đích}}(x)/p_{\text{nguồn}}(x)$,
ước lượng bằng một bộ phân loại phân biệt nguồn với đích (Tibshirani và cộng sự, 2019).

Cách này hấp dẫn vì **không cần một nhãn nào của miền đích**. Câu hỏi là nó có thay được ngân sách
nhãn ở Bước 7 không. Hãy chạy rồi kết luận, đừng đoán.

In [ ]:
rows = []
for v, (cal, pc, tgt, pt) in giu_lai.items():
    s_cal = np.abs(cal.capacity.values - pc)

    clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=.08, random_state=0)
    clf.fit(np.vstack([cal[FEATS].values, tgt[FEATS].values]),
            np.r_[np.zeros(len(cal)), np.ones(len(tgt))])
    pr = np.clip(clf.predict_proba(cal[FEATS].values)[:, 1], 1e-4, 1-1e-4)
    w_cal = pr / (1 - pr)
    pr2 = np.clip(clf.predict_proba(tgt[FEATS].values)[:, 1], 1e-4, 1-1e-4)
    w_te = pr2 / (1 - pr2)

    o = np.argsort(s_cal)
    ss, ws = s_cal[o], w_cal[o]
    cum = np.cumsum(ws)
    frac = cum[None, :] / (cum[-1] + w_te[:, None])      # có khối lượng ở vô cực cho điểm test
    j = np.where((frac >= NOM).any(1), (frac >= NOM).argmax(1), len(ss) - 1)
    qw = ss[j]
    c_w, w_w, _ = bao_phu(pt - qw, pt + qw, tgt.capacity.values)
    lo, hi = khoang_tuyet_doi(cal.capacity.values - pc, pt)
    c_0, w_0, _ = bao_phu(lo, hi, tgt.capacity.values)
    rows.append({'xe': v, 'khong_trong_so': c_0, 'co_trong_so': c_w,
                 'rong_khong': w_0, 'rong_co': w_w})
    print(f'xe {v}: không trọng số {c_0:.3f} -> có trọng số {c_w:.3f}', flush=True)

ts = pd.DataFrame(rows); ts.to_csv(RES / 'trong_so.csv', index=False)
print(f"\ntrung bình bao phủ: {ts.khong_trong_so.mean():.3f} -> {ts.co_trong_so.mean():.3f}")
print(f"trung bình độ rộng: {ts.rong_khong.mean():.2f} -> {ts.rong_co.mean():.2f} Ah")
print(f"số xe dưới mức danh nghĩa: {int((ts.khong_trong_so<NOM).sum())} -> "
      f"{int((ts.co_trong_so<NOM).sum())} trên {len(ts)}")

### Vì sao trọng số không cứu được

Nhìn bao phủ thôi thì tưởng thành công. Nhìn kèm độ rộng thì không: khoảng phình lên gấp mấy lần.
Phương pháp không hiệu chuẩn lại khoảng, nó chỉ thổi khoảng rộng tới mức không thể thiếu bao phủ nữa.

Gán trọng số sửa được bao phủ khi dịch chuyển **chỉ nằm ở đầu vào** và $p(y|x)$ giữ nguyên.
Hai chiếc xe cùng đời, cùng chạy một thành phố, có phân bố đường cong sạc gần như giống nhau, nên
tỉ số $w(x)$ nhiễu chứ không mang thông tin. Thứ khác nhau là **cách dung lượng phản ứng với đường
cong đó**, vì mỗi pack có cách ghép cell riêng, điện trở trong riêng, lịch sử riêng.

Đó là dịch chuyển ở **phân bố có điều kiện**, và không phép gán trọng số đầu vào nào sửa được.
Đây chính là lý do ngân sách nhãn ở Bước 7 là thứ không thể tránh, và cũng là đóng góp của bài báo.

## Bước 9: xuất bảng và hình cho bài báo

Mọi con số trong bài phải đọc ra từ các file này, **không gõ tay**. Gõ tay là cách chắc chắn nhất
để bài báo nói một đằng, thí nghiệm ra một nẻo.

In [ ]:
xuat = {
    'diem_so.csv': 'RMSE, MAE, MAPE, R2 của 8 mô hình dưới 3 giao thức',
    'conformal_doixe.csv': 'bao phủ và độ rộng của 8 mô hình dưới 3 giao thức',
    'conformal_xaotron.csv': 'bao phủ và độ rộng của 3 điểm số khi hoán vị được',
    'kiem_dinh.csv': 'Friedman và Wilcoxon-Holm trên 20 lượt giữ xe',
    'ngan_sach.csv': 'bao phủ theo số phiên có nhãn k, hai trục dịch chuyển',
    'trong_so.csv': 'so sánh có và không gán trọng số dịch chuyển hiệp biến',
    'so_sanh_bo_dac_trung.csv': 'bộ 43 của bài báo so với bộ 167 mở rộng',
    'tam_quan_trong.csv': 'tầm quan trọng của 167 đặc trưng theo XGBoost',
}
for f, mo_ta in xuat.items():
    print(f'{"CÓ " if (RES / f).exists() else "THIẾU"} {f:24s} {mo_ta}')

cxv = cx[cx.giao_thuc == 'đổi xe']
print('\nBốn con số cho phần Tóm tắt của bài báo:')
print(f'  bao phủ khi hoán vị được                      : {ind.bao_phu.mean():.3f}')
print(f'  bao phủ trung bình khi đổi xe                 : '
      f'{cxv[cxv.model=="XGBoost"].bao_phu.mean():.3f}')
print(f'  xe tệ nhất khi đổi xe                         : '
      f'{cxv[cxv.model=="XGBoost"].bao_phu.min():.3f}')
print(f'  bao phủ khi đổi thời kỳ, chỉ hiệu chuẩn nguồn : '
      f'{ns[(ns.truc=="đổi thời kỳ")&(ns.cach=="Chỉ nguồn")].bao_phu.mean():.3f}')
dich = ns[(ns.truc=="đổi thời kỳ")&(ns.cach=="Chỉ đích")].groupby('k').bao_phu.mean()
dat = dich[dich >= NOM]
print(f'  k nhỏ nhất để cách Chỉ đích lấy lại mức danh nghĩa : '
      f'{int(dat.index[0]) if len(dat) else "chưa đạt trong khoảng k đã thử"}')

## Checklist trước khi nộp

| Mục | Đạt chưa |
|---|---|
| Phép thử xáo trộn cho bao phủ trong khoảng 0,89 tới 0,91 | |
| `assert` tập huấn luyện và tập hiệu chuẩn rời nhau đã chạy | |
| Dùng `q_level(n)` chứ không phải `np.quantile(s, 0.9)` | |
| Bảng bao phủ có cột "xe tệ nhất", không chỉ có trung bình | |
| Mỗi $k$ được bốc lại 20 lần | |
| Mọi con số trong bài báo đọc từ `results/`, không gõ tay | |
| Audit Log có ít nhất 3 dòng trong tuần | |

## Việc tiếp theo của Sinh viên B

1. Viết Mục 4 (Method) và Mục 5 (Results) của bài báo từ Bước 3 tới Bước 8.
2. Chuẩn bị trả lời câu phản biện chắc chắn sẽ có: *"đổi xe hầu như không làm RMSE tệ đi, vậy vì sao
   các bạn còn nói có dịch chuyển?"*. Câu trả lời nằm ở cột xe tệ nhất của Bước 5, không nằm ở RMSE.
3. Chạy lại toàn bộ notebook từ đầu với kernel sạch, xác nhận không ô nào lỗi, ghi lại thời gian
   chạy thật, rồi nộp kèm `results/`.